# 1. Churn Data - Neural Network Analysis

### Data Preprocessing (Normalization, Encoding, Correlation Handling)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load dataset
df = pd.read_csv("churn-bigml-20.csv")

# Convert categorical variables to numeric
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

# Normalize numerical variables
scaler = StandardScaler()
X = df.drop('Churn', axis=1)
y = df['Churn']

X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

# Remove highly correlated variables (>0.8)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

drop_cols = [column for column in upper.columns if any(upper[column] > 0.8)]
X = X.drop(columns=drop_cols)

print("Dropped correlated columns:", drop_cols)

Dropped correlated columns: ['Number vmail messages', 'Total day charge', 'Total eve charge', 'Total night charge', 'Total intl charge']


## Neural Network Model

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Neural Network Model
nn_model = MLPClassifier(hidden_layer_sizes=(5,), activation='relu', max_iter=500, random_state=42)
nn_model.fit(X_train, y_train)

# Predictions
y_pred = nn_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8905472636815921
              precision    recall  f1-score   support

       False       0.91      0.97      0.94       173
        True       0.67      0.43      0.52        28

    accuracy                           0.89       201
   macro avg       0.79      0.70      0.73       201
weighted avg       0.88      0.89      0.88       201



c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## Variable Importance

In [3]:
# Calculate feature importance based on weights
importance = np.sum(np.abs(nn_model.coefs_[0]), axis=1)

feature_importance = pd.Series(importance, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print(feature_importance)

Total eve minutes                      2.348289
Total day minutes                      2.237510
Customer service calls                 2.199795
Total night calls                      2.091286
Number of Calls to customer service    2.017941
State                                  1.968333
Area code                              1.888768
Total intl minutes                     1.827203
International plan                     1.683653
Total eve calls                        1.316517
Voice mail plan                        1.306555
Total intl calls                       1.267739
Total day calls                        1.241136
Account length                         1.176494
Total night minutes                    1.046712
dtype: float64


# 2. ClassifyRisk Data

### Neural Network (Age → Income)

In [6]:
import pandas as pd
from sklearn.neural_network import MLPClassifier

# Load data
df2 = pd.read_csv("ClassifyRisk.csv")

# Clean column names
df2.columns = df2.columns.str.strip().str.lower()

# Convert income into categories (example: High / Low)
median_income = df2['income'].median()
df2['income_class'] = df2['income'].apply(lambda x: 1 if x > median_income else 0)

# Define X and y
X = df2[['age']]
y = df2['income_class']

# Train model
nn_model = MLPClassifier(hidden_layer_sizes=(1,), max_iter=500, random_state=42)
nn_model.fit(X, y)

# Print weights
print("Weights Input→Hidden:", nn_model.coefs_[0])
print("Weights Hidden→Output:", nn_model.coefs_[1])

Weights Input→Hidden: [[-0.35168661]]
Weights Hidden→Output: [[0.71606589]]


# 3.  Association Rules (Churn Data)

### a. Generate Rules

In [12]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df_rules = pd.read_csv("churn-bigml-20.csv")

# 1. CHECK COLUMN NAMES (Uncomment if names still fail)
# print(df_rules.columns)

# 2. Update these strings to match the output of print(df_rules.columns)
cols = ['Voice mail plan', 'International plan', 'Customer service calls', 'Churn']
df_rules = df_rules[cols]

# 3. Convert to binary/boolean for Apriori
df_rules = pd.get_dummies(df_rules.astype(str)).astype(bool)

# 4. Generate frequent itemsets
frequent_itemsets = apriori(df_rules, min_support=0.01, use_colnames=True)

# 5. Generate rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.05)

print(rules.head())

                antecedents                 consequents  antecedent support  \
0      (Voice mail plan_No)     (International plan_No)            0.716642   
1   (International plan_No)        (Voice mail plan_No)            0.920540   
2      (Voice mail plan_No)    (International plan_Yes)            0.716642   
3  (International plan_Yes)        (Voice mail plan_No)            0.079460   
4      (Voice mail plan_No)  (Customer service calls_0)            0.716642   

   consequent support   support  confidence      lift  representativity  \
0            0.920540  0.662669    0.924686  1.004504               1.0   
1            0.716642  0.662669    0.719870  1.004504               1.0   
2            0.079460  0.053973    0.075314  0.947817               1.0   
3            0.716642  0.053973    0.679245  0.947817               1.0   
4            0.212894  0.149925    0.209205  0.982674               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  


###  b. Rule with Highest Lift

In [13]:
best_rule = rules.sort_values(by='lift', ascending=False).iloc[0]
print(best_rule)

antecedents           (Voice mail plan_Yes, Churn_True)
consequents                    (International plan_Yes)
antecedent support                             0.022489
consequent support                              0.07946
support                                        0.010495
confidence                                     0.466667
lift                                           5.872956
representativity                                    1.0
leverage                                       0.008708
conviction                                     1.726012
zhangs_metric                                  0.848817
jaccard                                        0.114754
certainty                                       0.42063
kulczynski                                     0.299371
Name: 234, dtype: object


In [15]:
df = pd.read_csv("churn-bigml-20.csv")

# Select correct columns (fixed names)
df = df[['Voice mail plan', 'International plan', 'Customer service calls', 'Churn']]

# Rename columns (optional but recommended)
df.columns = ['VMail Plan', 'Intl Plan', 'CustServ Calls', 'Churn']

# Convert to categorical and dummy variables
df = pd.get_dummies(df.astype(str))

from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True)
rules_conf = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.05)

print(rules_conf.head())

       antecedents         consequents  antecedent support  \
0  (VMail Plan_No)      (Intl Plan_No)            0.716642   
1   (Intl Plan_No)     (VMail Plan_No)            0.920540   
2  (Intl Plan_Yes)     (VMail Plan_No)            0.079460   
3  (VMail Plan_No)     (Intl Plan_Yes)            0.716642   
4  (VMail Plan_No)  (CustServ Calls_0)            0.716642   

   consequent support   support  confidence      lift  representativity  \
0            0.920540  0.662669    0.924686  1.004504               1.0   
1            0.716642  0.662669    0.719870  1.004504               1.0   
2            0.716642  0.053973    0.679245  0.947817               1.0   
3            0.079460  0.053973    0.075314  0.947817               1.0   
4            0.212894  0.149925    0.209205  0.982674               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
0  0.002972    1.055056       0.015825  0.680000   0.052183    0.822278  
1  0.002972    1.011523     

In [16]:
rules_conf['confidence_diff'] = rules_conf['confidence'] - rules_conf['consequent support']

rules_cd_40 = rules_conf[rules_conf['confidence_diff'] >= 0.40]

print(rules_cd_40[['antecedents','consequents','confidence','consequent support','confidence_diff']])

                          antecedents   consequents  confidence  \
78                 (CustServ Calls_5)  (Churn_True)    0.647059   
312  (CustServ Calls_5, Intl Plan_No)  (Churn_True)    0.571429   

     consequent support  confidence_diff  
78             0.142429          0.50463  
312            0.142429          0.42900  


In [17]:
# Select one rule to verify
rule = rules_cd_40.iloc[0]

print("Confidence:", rule['confidence'])
print("Consequent Support:", rule['consequent support'])
print("Confidence Difference:", rule['confidence_diff'])

Confidence: 0.6470588235294118
Consequent Support: 0.1424287856071964
Confidence Difference: 0.5046300379222154


In [18]:
rules_cd_30 = rules_conf[rules_conf['confidence_diff'] >= 0.30]

print(rules_cd_30[['antecedents','consequents','confidence','confidence_diff']])

                                         antecedents  \
78                                (CustServ Calls_5)   
186                (VMail Plan_No, CustServ Calls_4)   
236                     (VMail Plan_Yes, Churn_True)   
312                 (CustServ Calls_5, Intl Plan_No)   
315                               (CustServ Calls_5)   
324                   (Churn_True, CustServ Calls_1)   
354                   (Churn_True, CustServ Calls_0)   
429  (Intl Plan_No, VMail Plan_No, CustServ Calls_4)   
433                (VMail Plan_No, CustServ Calls_4)   

                       consequents  confidence  confidence_diff  
78                    (Churn_True)    0.647059         0.504630  
186                   (Churn_True)    0.461538         0.319110  
236                (Intl Plan_Yes)    0.466667         0.387206  
312                   (Churn_True)    0.571429         0.429000  
315     (Intl Plan_No, Churn_True)    0.470588         0.356645  
324                (Intl Plan_Yes)    0.434

In [19]:
rules_conf['deployability'] = rules_conf['support'] * rules_conf['confidence']

best_deploy_rule = rules_conf.sort_values(by='deployability', ascending=False).iloc[0]

print(best_deploy_rule)

antecedents            (Churn_False)
consequents           (Intl Plan_No)
antecedent support          0.857571
consequent support           0.92054
support                     0.806597
confidence                  0.940559
lift                        1.021748
representativity                 1.0
leverage                    0.017168
conviction                  1.336802
zhangs_metric               0.149442
jaccard                     0.830247
certainty                   0.251946
kulczynski                   0.90839
confidence_diff              0.02002
deployability               0.758652
Name: 47, dtype: object


In [20]:
# Already satisfied with same settings
rules_conf = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.05)

In [21]:
rules_conf['confidence_ratio'] = rules_conf['confidence'] / rules_conf['consequent support']

rules_cr_40 = rules_conf[rules_conf['confidence_ratio'] >= 1.40]

print(rules_cr_40[['antecedents','consequents','confidence','confidence_ratio']])

                                     antecedents  \
32                            (CustServ Calls_5)   
57                               (Intl Plan_Yes)   
58                                  (Churn_True)   
76                                  (Churn_True)   
77                            (CustServ Calls_4)   
..                                           ...   
458                                 (Churn_True)   
459  (Intl Plan_Yes, Churn_False, VMail Plan_No)   
463                 (Intl Plan_Yes, Churn_False)   
467            (VMail Plan_No, CustServ Calls_2)   
469                           (CustServ Calls_2)   

                                          consequents  confidence  \
32                                   (VMail Plan_Yes)    0.411765   
57                                       (Churn_True)    0.358491   
58                                    (Intl Plan_Yes)    0.200000   
76                                 (CustServ Calls_4)    0.126316   
77                            

In [22]:
intl_rules = rules_cr_40[rules_cr_40['antecedents'].astype(str).str.contains('Intl Plan')]

selected_rule = intl_rules.iloc[0]

print(selected_rule)

antecedents           (Intl Plan_Yes)
consequents              (Churn_True)
antecedent support            0.07946
consequent support           0.142429
support                      0.028486
confidence                   0.358491
lift                         2.516981
representativity                  1.0
leverage                     0.017168
conviction                   1.336802
zhangs_metric                0.654723
jaccard                      0.147287
certainty                    0.251946
kulczynski                   0.279245
confidence_ratio             2.516981
Name: 57, dtype: object


In [23]:
gri_rules = rules_conf[rules_conf['consequents'].astype(str).str.contains('Churn')]

# Apply reasonable thresholds
gri_rules = gri_rules[(gri_rules['support'] >= 0.01) & (gri_rules['confidence'] >= 0.05)]

print(gri_rules[['antecedents','consequents','support','confidence']])

                                          antecedents  \
16                                    (VMail Plan_No)   
17                                    (VMail Plan_No)   
33                                   (VMail Plan_Yes)   
35                                   (VMail Plan_Yes)   
48                                     (Intl Plan_No)   
..                                                ...   
522                                (CustServ Calls_3)   
525  (VMail Plan_Yes, Intl Plan_No, CustServ Calls_4)   
527                  (Intl Plan_No, CustServ Calls_4)   
528                (VMail Plan_Yes, CustServ Calls_4)   
529                                (CustServ Calls_4)   

                                     consequents   support  confidence  
16                                 (Churn_False)  0.596702    0.832636  
17                                  (Churn_True)  0.119940    0.167364  
33                                 (Churn_False)  0.260870    0.920635  
35                     

In [24]:
print("Total Apriori Rules:", len(rules_conf))
print("GRI-like Rules:", len(gri_rules))

Total Apriori Rules: 530
GRI-like Rules: 200
